# Preparation

In [106]:
import json
import pandas as pd
from collections import Counter
import requests
import re
from pathlib import Path

In [66]:
home = Path.home()

# Functions

In [42]:
TOKEN_RE = re.compile(r"\d+|[^\W\d_]+|[.,/:;()\[\]-]")

def skeleton(text):
    out = []
    prev = None

    for tok in TOKEN_RE.findall(text):
        if tok[0].isdigit():
            kind = "N"
        elif tok[0].isalpha():
            kind = "W"
        else:
            kind = tok

        # Collapse consecutive words
        if kind == "W" and prev == "W":
            continue

        out.append(kind)
        prev = kind

    return "".join(out)

In [57]:
def walk_keys(obj, prefix=""):
    if isinstance(obj, dict):
        for key, value in obj.items():
            path = f"{prefix}.{key}" if prefix else key
            yield path
            yield from walk_keys(value, path)

# Analysis

## Initial data

In [67]:
structure_counter = Counter()


objects = []

with open(f"{home}/code/data/shbd/shb.jsonld.lines", "r", encoding="utf-8") as f:
	objects = [json.loads(row) for row in f]

skeleton_notes = []

example = {}
	
print(len(objects))

for object in objects:
	entity = object["@graph"][1]
	if "hasNote" in entity:
		note = entity["hasNote"][0]["label"]
		pattern = skeleton(note)
		skeleton_notes.append(pattern)
		structure_counter.update([pattern])

		example.setdefault(pattern, note)

print(len(skeleton_notes))
print(*skeleton_notes[:3], sep="\n")

79114
79024
W,W,W-W:WN:(W).-W:W,WN-N,N,N,W.N-N.-W.W-W,WN-N
W,W.W,W.-W:W,WN-N,N,N:N,W.N-N
W,W,W:W.-W:W,WN-N,N:N,W.N-N


### Count properties

In [61]:
property_counts = Counter()
subject_counts = Counter()

for object in objects:
	entity = object["@graph"][1]
	property_counts.update(walk_keys(entity))
      
	subjects = entity.get("instanceOf", {}).get("subject", [])
    
	subject_counts.update(
          subject["@id"]
              for subject in subjects
                    )

In [62]:
for key, count in property_counts.most_common():
    print(f"{count:>5}  {key}")

79114  @id
79114  @type
79114  category
79114  publication
79114  isPartOf
79114  part
79114  associatedMedia
79114  marc:primaryProvisionActivity
79114  marc:primaryProvisionActivity.year
79114  marc:primaryProvisionActivity.@type
79114  instanceOf
79114  instanceOf.@id
79024  hasNote


### Inspect structure of descriptions

In [53]:
print("| Count | Pattern | Example |")
print("|------:|---------|---------|")

for pattern, count in structure_counter.most_common(20):
    ex = example[pattern].replace("|", "\\|")  # Escape pipes if any
    print(f"| {count} | `{pattern}` | {ex} |")

| Count | Pattern | Example |
|------:|---------|---------|
| 842 | `W,W.,W.(WN,W.N-N.)` | Dalgren, L., Ur den nyaste tyska Arndtlitteraturen. (HT 1922, s. 247-249.) |
| 569 | `W,W.,W.(W.N(N),W.N-N.)` | Brulin, H., Das schwedische Archivwesen. (Archivalische Zeitschr. 38 (1929),s. 151-177.) |
| 566 | `W,W,W.(WN,W.N-N.)` | Lundberg, Erik, Nyare forskning över svensk byggnadshistoria. (Rig 1932,s. 105-127.) |
| 504 | `W,W,W.-WN.NN.` | Jagerskiold, Stig, Svea hovrätt jubilerar. - SvD 17.2 1964. |
| 417 | `W,W.,W.(WN(N),W.N-N.)` | Berghman, A., Heraldisk litteratur. (MRÄ 4 (1935), s. 9-38.) |
| 414 | `W,W,W.(W.N(N),W.N-N.)` | Floderus, Erik, Våra äldsta mynt. (Kooperatören. 17 (1930), s. 120-126.) |
| 321 | `W,W,W.(WN(N),W.N-N.)` | Söderberg, Bengt, Gotländska glasmålningar med länsherrevapen. (GA 5(1933), s. 37-44.) |
| 294 | `W,W,W.-WN(N),W.N-N.` | Åkerman, Sune, Projects and research priorities. - Historisk tidskrift 90 (1970),  s. 47-67. |
| 266 | `W,W,W.(WN/NN.)` | Leide, Arvid, Danie

## Enriched data

In [137]:
objects = []

with open(f"{home}/code/data/shbd/shb-cleaned-with-subjects.jsonld.lines", "r", encoding="utf-8") as f:
	objects = [json.loads(row) for row in f]
	
print(len(objects))


79060


In [138]:
property_counts = Counter()
subject_counts = Counter()

instances = []
for object in objects:
	entity = object["@graph"]["@graph"][1]
	property_counts.update(walk_keys(entity))
      
	subjects = entity.get("instanceOf", {}).get("subject", [])
	subject_counts.update(
          subject["@id"]
              for subject in subjects
                    )
	
	instances.append(entity)

print(instances[:3])

[{'@id': 'https://libris-qa.kb.se/dataset/shb/1#it', '@type': 'PhysicalResource', 'category': [{'@id': 'https://id.kb.se/term/saobf/ComponentPart'}, {'@id': 'https://id.kb.se/term/saobf/Print'}], 'instanceOf': {'@type': 'Monograph', 'category': [{'@id': 'https://id.kb.se/term/rda/Text'}]}, 'hasTitle': {'@type': 'Title', 'mainTitle': 'Malmö-litteratur', 'subtitle': 'bibliografiska noteringar för år1975 : (med tillägg från föregående år).'}, 'responsibilityStatement': 'Andersson, Per', 'partOf': [{'@type': 'Instance', 'label': 'Malmö, ISSN 0348-0909,44, 1976Även utg. i serien Malmö-litteratur, ISSN0348-0917', 'hasTitle': {'@type': 'Title', 'mainTitle': 'Malmö, ISSN 0348-0909,44, 1976'}, 'identifiedBy': {'@type': 'ISSN', 'value': '0348-0909'}, 'extent': [{'@type': 'Extent', 'label': None}]}], 'hasNote': [{'@type': 'Note', 'label': 'Fullständig beskrivning (OCR) ur SHBD: Andersson, Per, Malmö-litteratur : bibliografiska noteringar för år1975 : (med tillägg från föregående år). - I: Malmö, 

In [151]:
extent = [{"@id": i["@id"], "extent": i["extent"][0]["label"]} for i in instances if "extent" in i and i["extent"][0]["label"][0].isalpha()]
extent_df = pd.json_normalize(extent)
extent_df.info()
extent_df.head(2)

pd.DataFrame(extent_df.value_counts(subset=["extent"])).head(50)


<class 'pandas.DataFrame'>
RangeIndex: 21421 entries, 0 to 21420
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   @id     21421 non-null  str  
 1   extent  21421 non-null  str  
dtypes: str(2)
memory usage: 334.8 KB


,count
extent,
s. 4,178
s. 2,73
s. 1,32
s. 1-16,29
s. 1-8,26
s. 1-11,24
s. 1-12,22
s. 1-24,22
s. 1-7,22


### Inspect seriesStatement

In [139]:
series_membership = [{"@id": i["@id"], "seriesMembership": i["seriesMembership"][0]} for i in instances if "seriesMembership" in i]

print(series_membership[:3])    

[{'@id': 'https://libris-qa.kb.se/dataset/shb/10#it', 'seriesMembership': {'@type': 'Instance', 'label': 'Specialarbete / Bibliotekshögskolan, ISSN 0347-1128 ; 1976:158', 'hasTitle': {'@type': 'Title', 'mainTitle': 'Specialarbete / Bibliotekshögskolan, ISSN 0347-1128 ; 1976:158'}, 'identifiedBy': {'@type': 'ISSN', 'value': '0347-1128'}}}, {'@id': 'https://libris-qa.kb.se/dataset/shb/28#it', 'seriesMembership': {'@type': 'Instance', 'label': 'Acta Bibliothecae regiae Stockholmiensis,ISSN 0065-1060 ; 28', 'hasTitle': {'@type': 'Title', 'mainTitle': 'Acta Bibliothecae regiae Stockholmiensis,ISSN 0065-1060 ; 28'}, 'identifiedBy': {'@type': 'ISSN', 'value': '0065-1060'}}}, {'@id': 'https://libris-qa.kb.se/dataset/shb/52#it', 'seriesMembership': {'@type': 'Instance', 'label': 'Småskriftserien / Föreningen Gamla Vadstena, ISSN 0426-6587 ; 22', 'hasTitle': {'@type': 'Title', 'mainTitle': 'Småskriftserien / Föreningen Gamla Vadstena, ISSN 0426-6587 ; 22'}, 'identifiedBy': {'@type': 'ISSN', 'val

In [140]:
series_df = pd.json_normalize(series_membership)
series_df.info()
series_df.head(2)

<class 'pandas.DataFrame'>
RangeIndex: 5491 entries, 0 to 5490
Data columns (total 9 columns):
 #   Column                                    Non-Null Count  Dtype 
---  ------                                    --------------  ----- 
 0   @id                                       5491 non-null   str   
 1   seriesMembership.@type                    5491 non-null   str   
 2   seriesMembership.label                    5491 non-null   str   
 3   seriesMembership.hasTitle.@type           5433 non-null   str   
 4   seriesMembership.hasTitle.mainTitle       5433 non-null   str   
 5   seriesMembership.identifiedBy.@type       33 non-null     str   
 6   seriesMembership.identifiedBy.value       33 non-null     str   
 7   seriesMembership.responsibilityStatement  84 non-null     str   
 8   seriesMembership.extent                   736 non-null    object
dtypes: object(1), str(8)
memory usage: 386.2+ KB


,@id,seriesMembership.@type,seriesMembership.label,seriesMembership.hasTitle.@type,seriesMembership.hasTitle.mainTitle,seriesMembership.identifiedBy.@type,seriesMembership.identifiedBy.value,seriesMembership.responsibilityStatement,seriesMembership.extent
0,https://libris-qa.kb.se/dataset/shb/10#it,Instance,"Specialarbete / Bibliotekshögskolan, ISSN 0347...",Title,"Specialarbete / Bibliotekshögskolan, ISSN 0347...",ISSN,0347-1128,NaN,NaN
1,https://libris-qa.kb.se/dataset/shb/28#it,Instance,"Acta Bibliothecae regiae Stockholmiensis,ISSN ...",Title,"Acta Bibliothecae regiae Stockholmiensis,ISSN ...",ISSN,0065-1060,NaN,NaN


In [141]:
most_common_titles = pd.DataFrame(series_df.value_counts(subset=["seriesMembership.hasTitle.mainTitle"]))

most_common_titles.head(20)

,count
seriesMembership.hasTitle.mainTitle,
GHT 6/4 1957,6
Särtr. ur Vasabladet,6
Skolminnen,5
Festskr. utg. av Teol. fak. i Upps. 1941,5
Hembygdsböckerna,4
UNT 1957: julnr,4
1971,3
StT 1925: 10/5,3
Memoarer och resor,3


### Count properties and subjects

#### Properties

In [113]:
for key, count in property_counts.most_common():
    print(f"{count:>5}  {key}")

79060  @id
79060  @type
79060  category
79060  instanceOf
79060  instanceOf.@type
79060  instanceOf.category
78970  hasTitle
78970  hasTitle.@type
78970  hasTitle.mainTitle
78970  hasNote
78322  instanceOf.subject
67752  responsibilityStatement
41964  extent
23505  seriesMembership
 6222  partOf
 3141  hasTitle.subtitle


#### Subjects

In [110]:
for key, count in subject_counts.most_common():

    print(f"{count:>5}  {key}")

# Random stuff

In [ ]:
headers = {"Accept": "application/ld+json"}

query_string = f"instanceType:PhysicalResource title:({shbd_prepepd['mainTitle']}) title:({shbd_prepepd['subtitle']}) contributor:{shbd_prepepd['responsibility_statement']}* {shbd_prepepd['part_of_issn']} {shbd_prepepd['issn_from_note']}"


params = {"_q": "title:Hembergska+huset+i+Simrishamn contributor:Ehrnberg, G.*",
          #"_embellished": "false", Den här verkar inte göra något
          "_lens": "chips", # Den här behöver vara i plural
          "_stats": "false",
          "limit": 10}

res = requests.get("http://libris.kb.se/find?", params = params, headers=headers)
res.raise_for_status()
print(res.url)

print("Status:", res.status_code)
print("Number of results:", res.json()["totalItems"])
print("\nResult keys:", *res.json().keys(), sep=", ")

# Var finns den vanliga bibliografiska datan?
records = res.json()["items"]
print("\nItem keys:", *records[0].keys(), sep=", ")

print(records[0])


http://libris.kb.se/find?_q=title%3AHembergska%2Bhuset%2Bi%2BSimrishamn+contributor%3AEhrnberg%2C+G.%2A&_lens=chips&_stats=false&limit=10
Status: 200
Number of results: 1

Result keys:, @type, @id, search, itemOffset, itemsPerPage, totalItems, first, last, items, maxItems, @context

Item keys:, @type, meta, _categoryByCollection, @id, @reverse, hasTitle, language, contribution, reverseLinks
{'@type': 'Monograph', 'meta': {'mainEntity': {'@id': 'https://libris.kb.se/vc55wfk63mjdwll#work'}, '@type': 'VirtualRecord', '@id': 'https://libris.kb.se/vc55wfk63mjdwll#work-record'}, '_categoryByCollection': {'@none': [{'@type': 'ContentType', 'meta': {'mainEntity': {'@id': 'https://id.kb.se/term/rda/Text'}, '@type': 'Record', '@id': 'https://libris.kb.se/pnpsnkg0r5b6x092'}, '@id': 'https://id.kb.se/term/rda/Text', 'sameAs': [{'@id': 'https://id.kb.se/term/rda/content/text'}, {'@id': 'https://id.kb.se/term/rda/content/txt'}], 'prefLabelByLang': {'sv': 'Text', 'en': 'Text'}, 'code': 'txt', 'inSche

In [104]:
print(res.json()["stats"])

{'_predicates': [], 'sliceByDimension': {'librissearch:instanceType': {'dimension': 'librissearch:instanceType', 'observation': [{'totalItems': 1, 'view': {'@id': '/find?_q=title:Hembergska%2Bhuset%2Bi%2BSimrishamn+contributor:Ehrnberg%2C+G.*+instanceType:PhysicalResource'}, 'object': {'@id': 'PhysicalResource', '@type': 'Class', 'subClassOf': [{'@id': 'https://id.kb.se/vocab/Instance'}, {'@id': 'http://purl.org/dc/terms/PhysicalResource'}, {'@type': 'Restriction', 'onProperty': {'@id': 'https://id.kb.se/vocab/category'}, 'owl:onClass': {'hasValue': {'@id': 'https://id.kb.se/term/saobf/PhysicalForm'}, 'onProperty': {'@id': 'https://id.kb.se/vocab/broaderTransitive'}}, 'owl:minQualifiedCardinality': 1}], 'isDefinedBy': {'@id': 'https://id.kb.se/vocab/'}, 'labelByLang': {'en': 'Physical resource', 'sv': 'Fysisk resurs'}}}], 'maxItems': 100, '_connective': 'AND'}, 'librissearch:instanceCategory': {'dimension': 'librissearch:instanceCategory', 'observation': [{'totalItems': 1, 'view': {'@i

In [50]:
import re
rest = "Swedenborg : sökaren i naturens och andens värld :hans verk och efterföljd / Carl / Hej"
#rest = "Egerbladh, Ossian, Ur Lappmarkens bebyggelsehistoria. Umeå. 1-8. Se SHB 1961/70:7809.9 : Barsele : minnesskrift med anledning av byns tvåhundraåriga tillvaro.1970. 98 s.10 : Stensele 1741-1860 : de hundra äldsta nybyggesupptagningarna.1972. 237 s.11 : Fyra gamla Lyckselebyar : Björksele, Brattfors, Falträsk, Vägsele :denna utredning har utförts med anledning av Lycksele sockens 300-årsjubileum. 1973. 94 s. : ill."
subtitle = ""
title= ""
if ' : ' in rest:
	title, rest = rest.split(' : ', 1)
	print (title)
	print(rest)

	if ':' in rest:
		parts =  re.split(r" ([./])", rest, maxsplit=1)
		subtitle = parts[0]
		if len(parts) > 1:
			print(parts)
			rest = "".join(parts[1:])
print()
print(title)
print(subtitle)
print(rest)



Swedenborg
sökaren i naturens och andens värld :hans verk och efterföljd / Carl / Hej
['sökaren i naturens och andens värld :hans verk och efterföljd', '/', ' Carl / Hej']

Swedenborg
sökaren i naturens och andens värld :hans verk och efterföljd
/ Carl / Hej
